# 04 — Risk Measures: VaR, CVaR & Exceedance Backtesting

**What this notebook does:**

VaR and CVaR were already computed **per-window** inside the rolling loop (notebook 02), using the per-window NIG distribution and the affine transform:

$$\text{VaR}_\alpha = \mu_{t+1} + \sigma_{t+1} \cdot F^{-1}_{\text{NIG}_t}(1-\alpha)$$

This notebook loads those pre-computed risk measures and performs backtesting:

1. VaR exceedance counting with binomial $p$-values 
2. Color-coded results (green/red/blue per professor's convention)
3. VaR vs actual returns visualisation
4. CVaR summary (Expected Shortfall — required by Basel III)
5. Christoffersen independence test on hit sequences

In [ ]:
# Imports

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from scipy import stats
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "iframe"

import warnings
warnings.filterwarnings("ignore")

from src.data_utils import load_processed, save_processed
from src.assessment import (
    count_exceedances, binomial_pvalue, pvalue_color,
    christoffersen_test,
)

print("Imports OK")


In [ ]:
# Load pre-computed risk measures from notebook 02

var99 = load_processed("var99.parquet")
var95 = load_processed("var95.parquet")
cvar99 = load_processed("cvar99.parquet")
cvar95 = load_processed("cvar95.parquet")

TICKERS = list(var99.columns)

# Load full rolling results for actual returns
all_results = {}
for ticker in TICKERS:
    safe = ticker.replace("^", "").replace("=", "_")
    all_results[ticker] = load_processed(f"rolling_{safe}.parquet")

N = len(var99)  # assessment window size

print(f"Loaded: {len(TICKERS)} assets, {N} prediction dates")
print(f"Date range: {var99.index[0].date()} → {var99.index[-1].date()}")
print(f"Tickers: {TICKERS}")

# Quick sanity check
assert (var99 < 0).all().all(), "VaR 99% should all be negative"
assert (var95 < 0).all().all(), "VaR 95% should all be negative"
print("\nAll VaR values negative: PASS")

---
## 1. VaR Exceedance Backtesting

An exceedance occurs when the actual return falls below VaR (Kaufman slide 28):

$$e_t = \begin{cases} 1 & \text{if } r_t < \text{VaR}^t_\alpha \\ 0 & \text{otherwise} \end{cases}$$

The total exceedance count $E = \sum e_t \sim \text{Binomial}(n, 1-\alpha)$. We compute a two-sided binomial $p$-value.

**Color coding:**
- **Green:** $p \geq 0.05$ — model passes
- **Red:** $p < 0.05$ and exceedances > expected — risk understated
- **Blue:** $p < 0.05$ and exceedances < expected — risk overstated

In [ ]:
# Count exceedances and compute binomial p-values

LEVELS = [0.99, 0.95]
var_data = {0.99: var99, 0.95: var95}

results_table = []

for ticker in TICKERS:
    actual = all_results[ticker]["return"].values

    for level in LEVELS:
        var_s    = var_data[level][ticker].values
        exceed   = count_exceedances(actual, var_s)
        expected = N * (1 - level)
        pval     = binomial_pvalue(N, exceed, level)
        color    = pvalue_color(pval, exceed, expected)

        results_table.append({
            "Ticker":      ticker,
            "VaR level":   f"{int(level*100)}%",
            "Expected":    int(expected),
            "Exceedances": exceed,
            "Excess":      exceed - int(expected),
            "p-value":     round(pval, 4),
            "Result":      color.upper(),
        })

results_df = pd.DataFrame(results_table)
print("VaR Exceedance Backtesting Results\n")
print(results_df.to_string(index=False))

In [ ]:
# Color-coded p-value table

color_map = {"GREEN": "#2dc653", "RED": "#e63946", "BLUE": "#457b9d"}
font_map  = {"GREEN": "#144a23", "RED": "#5c0a10", "BLUE": "#0d2b40"}

def make_pvalue_table(df, title):
    fig = go.Figure(data=[go.Table(
        header=dict(
            values=["Ticker", "Expected", "Exceedances",
                    "Excess", "p-value", "Result"],
            fill_color="#2c3e50",
            font=dict(color="white", size=12),
            align="center", height=32,
        ),
        cells=dict(
            values=[
                df.index.tolist(),
                df["Expected"].tolist(),
                df["Exceedances"].tolist(),
                df["Excess"].tolist(),
                df["p-value"].tolist(),
                df["Result"].tolist(),
            ],
            fill_color=[
                ["#f8f9fa"] * len(df),
                ["#f8f9fa"] * len(df),
                ["#f8f9fa"] * len(df),
                ["#f8f9fa"] * len(df),
                ["#f8f9fa"] * len(df),
                [color_map[r] for r in df["Result"]],
            ],
            font=dict(
                color=[
                    ["#2c3e50"] * len(df)] * 5 + [
                    [font_map[r] for r in df["Result"]],
                ],
                size=12,
            ),
            align="center", height=28,
        ),
    )])
    fig.update_layout(
        title=title, template="plotly_white",
        height=260, margin=dict(t=50, b=10, l=10, r=10),
    )
    return fig

In [ ]:
# VaR 99% backtesting table

var99_df = results_df[results_df["VaR level"] == "99%"].set_index("Ticker")
fig99 = make_pvalue_table(var99_df, "VaR 99% Backtesting Results")
fig99.write_html("../outputs/figures/04_pvalue_table_99.html")
fig99.show()
print("Table saved")


In [ ]:
# VaR 95% backtesting table

var95_df = results_df[results_df["VaR level"] == "95%"].set_index("Ticker")
fig95 = make_pvalue_table(var95_df, "VaR 95% Backtesting Results")
fig95.write_html("../outputs/figures/04_pvalue_table_95.html")
fig95.show()
print("Table saved")

---
## 2. VaR vs Actual Returns

In [ ]:
# VaR vs actual returns for all assets

colors_assets = dict(zip(TICKERS, px.colors.qualitative.Plotly))

fig = make_subplots(
    rows=len(TICKERS), cols=1,
    shared_xaxes=True,
    subplot_titles=TICKERS,
    vertical_spacing=0.06,
)

for i, ticker in enumerate(TICKERS):
    actual = all_results[ticker]["return"]
    v99    = var99[ticker]
    v95    = var95[ticker]
    exceed_mask = actual.values < v99.values

    # Actual returns
    fig.add_trace(go.Scatter(
        x=actual.index, y=actual.values,
        mode="lines", name=f"{ticker} returns",
        line=dict(color=colors_assets[ticker], width=0.7),
        showlegend=False,
    ), row=i+1, col=1)

    # VaR 99%
    fig.add_trace(go.Scatter(
        x=v99.index, y=v99.values,
        mode="lines",
        name="VaR 99%" if i == 0 else None,
        showlegend=(i == 0),
        line=dict(color="#e63946", width=1.5),
    ), row=i+1, col=1)

    # VaR 95%
    fig.add_trace(go.Scatter(
        x=v95.index, y=v95.values,
        mode="lines",
        name="VaR 95%" if i == 0 else None,
        showlegend=(i == 0),
        line=dict(color="#c47000", width=1.2, dash="dot"),
    ), row=i+1, col=1)

    # Exceedances
    exc_dates  = actual.index[exceed_mask]
    exc_values = actual.values[exceed_mask]
    fig.add_trace(go.Scatter(
        x=exc_dates, y=exc_values,
        mode="markers",
        name="Exceedance" if i == 0 else None,
        showlegend=(i == 0),
        marker=dict(color="#040720", size=5, symbol="x"),
    ), row=i+1, col=1)

    fig.update_yaxes(title_text="Log return", row=i+1, col=1)

fig.update_layout(
    title="Actual Returns vs VaR 99% and VaR 95% — All Assets",
    height=1000, template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.04),
)
fig.write_html("../outputs/figures/04_var_vs_returns.html")
fig.show()
print("Figure saved")

---
## 3. Christoffersen Independence Test

Tests whether VaR exceedances cluster (serial dependence) or are independent. Clustering indicates the GARCH model has not fully captured volatility dynamics.

$p > 0.05$ means exceedances are independent (model passes).

In [ ]:
# Christoffersen independence test on hit sequences

christ_rows = []
for ticker in TICKERS:
    actual = all_results[ticker]["return"].values
    for level in LEVELS:
        var_s = var_data[level][ticker].values
        hits  = (actual < var_s).astype(int)
        result = christoffersen_test(hits)

        christ_rows.append({
            "Ticker": ticker,
            "VaR level": f"{int(level*100)}%",
            "Statistic": result["statistic"],
            "p-value": result["pvalue"],
            "Independent": "✓" if result["independent"] else "✗",
        })

christ_df = pd.DataFrame(christ_rows)
print("Christoffersen Independence Test\n")
print(christ_df.to_string(index=False))
print("\n✓ = exceedances are independent (p > 0.05)")
print("✗ = exceedances cluster — residual volatility dynamics remain")

---
## 4. CVaR (Expected Shortfall) Summary

CVaR = $E[R \mid R < \text{VaR}_\alpha]$ — the average loss in the worst scenarios. Required by Basel III as a supplement to VaR.

In [ ]:
# CVaR summary

cvar_rows = []
for ticker in TICKERS:
    cvar_rows.append({
        "Ticker": ticker,
        "CVaR 99% (mean)": round(cvar99[ticker].mean() * 100, 4),
        "CVaR 95% (mean)": round(cvar95[ticker].mean() * 100, 4),
        "CVaR 99% (worst)": round(cvar99[ticker].min() * 100, 4),
        "CVaR 95% (worst)": round(cvar95[ticker].min() * 100, 4),
    })

cvar_summary = pd.DataFrame(cvar_rows).set_index("Ticker")
print("CVaR Summary (% log returns --> negative = loss threshold)\n")
display(cvar_summary)
print("\nCVaR is always more negative than VaR —")
print("it represents the average loss in the worst scenarios, not just the threshold.")

---

In [ ]:
# Save results

save_processed(results_df, "exceedance_results.parquet")
save_processed(christ_df, "christoffersen_results.parquet")
save_processed(cvar_summary.reset_index(), "cvar_summary.parquet")
print("Saved: exceedance_results, christoffersen_results, cvar_summary")

In [ ]:
# Verification

print("Notebook 04 — Summary\n")
print(f"Assets: {len(TICKERS)}, Predictions: {N}")
print(f"\nVaR Exceedance Results:")
print(results_df[["Ticker", "VaR level", "Expected",
                   "Exceedances", "p-value", "Result"]].to_string(index=False))
print(f"\nChristoffersen Results:")
print(christ_df[["Ticker", "VaR level", "p-value",
                  "Independent"]].to_string(index=False))
print("\nProceed to notebook 05 (full assessment suite).")

---